# Next-Word Prediction with an LSTM

This notebook builds a small next-word prediction model from a few famous quotes. Each quote is turned into n-gram sequences ("the" → "greatest", "the greatest" → "glory", ...), padded, and used to train an LSTM that predicts the next word. It then generates text from a seed phrase. The original notebook stopped after creating the n-grams; this version completes the pipeline.

**How to run:** Open in Google Colab (CPU is fine) and run all cells.

_Rewritten from my original Colab notebook: `Untitled4.ipynb`._

## 1. Imports and seeds

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

np.random.seed(42)
tf.random.set_seed(42)

## 2. Dataset and tokenizer

In [ ]:
dataset = [
    "The greatest glory in living lies not in never falling, but in rising every time we fall.",
    "The way to get started is to quit talking and begin doing.",
    "Your time is limited, so don't waste it living someone else's life.",
    "The future belongs to those who believe in the beauty of their dreams.",
    "If you look at what you have in life, you'll always have more.",
    "If you set your goals ridiculously high and it's a failure, you will fail above everyone else's success.",
    "You may say I'm a dreamer, but I'm not the only one.",
]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(dataset)
total_words = len(tokenizer.word_index) + 1  # +1 because index 0 is padding
print("Vocabulary size:", total_words)

## 3. Build n-gram sequences

In [ ]:
input_sequences = []
for line in dataset:
    tokens = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(tokens)):
        input_sequences.append(tokens[: i + 1])

# Show the first few n-grams as words
for seq in input_sequences[:5]:
    print(" ".join(tokenizer.index_word[t] for t in seq))

max_len = max(len(s) for s in input_sequences)
padded = pad_sequences(input_sequences, maxlen=max_len, padding="pre")
X, y = padded[:, :-1], padded[:, -1]  # all words but the last -> the last word
print("X shape:", X.shape, "y shape:", y.shape)

## 4. Train the model

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(max_len - 1,)),
    layers.Embedding(total_words, 64),
    layers.LSTM(100),
    layers.Dense(total_words, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = model.fit(X, y, epochs=200, verbose=0)
print("Final training accuracy:", round(history.history["accuracy"][-1], 3))

## 5. Generate text

In [ ]:
def generate(seed_text, n_words=8):
    for _ in range(n_words):
        tokens = tokenizer.texts_to_sequences([seed_text])[0]
        tokens = pad_sequences([tokens], maxlen=max_len - 1, padding="pre")
        next_id = int(np.argmax(model.predict(tokens, verbose=0)[0]))
        seed_text += " " + tokenizer.index_word.get(next_id, "")
    return seed_text

print(generate("the greatest"))
print(generate("if you look"))
print(generate("the future"))